In [23]:
# Importing libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [24]:
# Loading the physics and supplementary data set and ignoring the lines beginning with '#'
data_frame_physics = pd.read_csv("../data/q1_q17_dr25_koi_2026.03.04_09.33.29.csv" , comment= "#")
data_frame_supplementary = pd.read_csv("../data/q1_q17_dr25_sup_koi_2026.02.14_00.59.48.csv" , comment="#")

In [25]:
# Left merging data from physics to supplementary 
joining_columns = data_frame_supplementary.columns.difference(data_frame_physics.columns).tolist()
joining_columns.append('kepid')
data_frame_merged = pd.merge(data_frame_physics, data_frame_supplementary[joining_columns], on="kepid", how="left")

In [26]:
# Filtering out the required columns
required_columns = [
    "kepid", # ID
    "koi_disposition", # Target
    "koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec", # Flags (High predictive power)
    "koi_period", "koi_duration", "koi_depth", "koi_ror", "koi_model_snr", # Transit Parameters
    "koi_prad", "koi_sma", "koi_incl", "koi_teq", "koi_insol", # Planetary Parameters
    "koi_steff", "koi_slogg", "koi_srad", "koi_smass" # Stellar Parameters
]

data_frame_merged = data_frame_merged[required_columns]

In [27]:
# Target encoding labels to floats
mapping = {"FALSE POSITIVE": 0, "CANDIDATE": 0.5, "CONFIRMED": 1}
data_frame_merged["Target"] = data_frame_merged["koi_disposition"].map(mapping)

In [28]:
# Dropping fully empty columns and empty values in rows
data_frame_merged = data_frame_merged.dropna(axis = 1, how = "all")
data_frame_merged = data_frame_merged.dropna()

In [29]:
# Dropping flag columns to make the model reliant on the physics features
flags_to_drop = ["koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec"]
data_frame_merged = data_frame_merged.drop(columns=flags_to_drop)

In [30]:
# Separating out unconfirmed "candidates"
data_frame_candidates = data_frame_merged[data_frame_merged["Target"] == 0.5].copy()
data_frame_binary = data_frame_merged[data_frame_merged["Target"] != 0.5].copy()

In [31]:
# Scaling the features while excluding the text columns and Target
standardizer = StandardScaler()
excluded_columns = ["kepid", "koi_disposition", "Target" ]
data_frame_excluded = data_frame_binary[data_frame_binary.columns.difference(excluded_columns)]
data_frame_standardized = standardizer.fit_transform(data_frame_excluded)

In [32]:
# Adding back the ID and Target Columns
data_frame_standardized = pd.DataFrame(data_frame_standardized, columns=data_frame_excluded.columns, index=data_frame_excluded.index)
data_frame_standardized["kepid"] = data_frame_binary["kepid"]
data_frame_standardized["Target"] = data_frame_binary["Target"]

# Splitting the data into train and test in a ration of 80 / 20 respectively
data_frame_train, data_frame_test = train_test_split(data_frame_standardized, test_size = 0.2, random_state = 42)